In [1]:
from IPython.display import display, Markdown, HTML, clear_output
from docx import Document
from docx.shared import Cm, Pt, Length, RGBColor
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.enum.table import WD_ALIGN_VERTICAL, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
import docx.oxml.shared
import pandas as pd
import json
import re
from Bio import Entrez
import xmltodict
import urllib.request
from urllib.request import urlopen
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')
%run Redmine_Access.ipynb
%run MyCancerGenome_parsing_final.ipynb
%run ACMG.ipynb

In [2]:
patient_info=getSampleInformationFromRedmine(65)

In [3]:
with open('DKS1002359122 Mutation report_06122023-120159.json') as json_file:
    data = json.load(json_file)       

In [4]:
def add_header_footer(document):
    for section in document.sections:
        # Set header
        header = section.header
        header_paragraph = header.paragraphs[0] if header.paragraphs else header.add_paragraph()
        header_paragraph.paragraph_format.left_indent = Cm(2.7) 
        header_run = header_paragraph.add_run()
        header_run.add_picture('G-KnowMe_Logo.png', width=Cm(5.4), height=Cm(2.71))
        # Set footer
        footer = section.footer
        footer_paragraph = footer.paragraphs[0] if footer.paragraphs else footer.add_paragraph()
        footer_paragraph.paragraph_format.left_indent = Cm(2.7)  
        footer_run = footer_paragraph.add_run("Private and confidential")
        footer_run.font.size = Pt(12)
        footer_run.font.name = 'Times New Roman'
        footer_run.italic = True

In [5]:
def add_internal_reference_table(document):
    centered_heading = document.add_paragraph("Table only for internal reference")
    centered_heading.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    centered_heading.runs[0].font.size = Pt(12)
    centered_heading.runs[0].font.name = 'Times New Roman'
    headings = ["Patient name", "Age", "Gender", "Diagnosis", "Comorbidities", "Specimen",
                "Histopathology in case of breast cancer/ovarian cancer etc.", "Metastatic status",
                "Known Infectious causes (for example HPV)?", "Treatment and progression/resistance history",
                "TMB status", "MSI status", "PD-L1 status", "HRR status", "MMR status",
                "Genes for clinical interpretation", "Additional comment/Family history etc."]
    table = document.add_table(rows=0, cols=2)
    table.style = 'Table Grid'
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(5.26)
    table.columns[1].width = Cm(12.7)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    for heading in headings:
        row = table.add_row().cells
        row[0].text = heading
        row[1].text = ""
        for cell in row:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.name = 'Times New Roman'
                    run.font.size = Pt(12)
    document.add_page_break()

In [6]:
def patient_information_table():
    sections = doc.sections
    for section in sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)
    table = doc.add_table(rows=1, cols=2)
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(5.1)
    table.columns[1].width = Cm(11.89)
    table.style = 'Table Grid'
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.cell(0, 0).text = 'Name:'
    table.cell(0, 1).text = patient_info.get('patient_name', '')or '' 
    table.add_row().cells[0].text = 'Age/Gender:'
    table.cell(1, 1).text = f"{patient_info.get('patient_age', '')}/{patient_info.get('patient_gender', '')}"or '' 
    table.add_row().cells[0].text = 'Referring Physician:'
    table.cell(2, 1).text = patient_info.get('referring_physician', '')or '' 
    table.add_row().cells[0].text = 'Referring Centre:'
    table.cell(3, 1).text = patient_info.get('referring_centre', '')or '' 
    table.add_row().cells[0].text = 'Specimen Type:'
    table.cell(4, 1).text = patient_info.get('specimen_type', '')or '' 
    table.add_row().cells[0].text = 'Sample ID:'
    table.cell(5, 1).text = patient_info.get('sample_id', '')or '' 
    table.add_row().cells[0].text = 'Patient ID:'
    table.cell(6, 1).text = patient_info.get('patient_id', '')or '' 
    table.add_row().cells[0].text = 'Date received:'
    table.cell(7, 1).text = patient_info.get('date_received', '')or '' 
    table.add_row().cells[0].text = 'Report Date:'
    table.cell(8, 1).text = patient_info.get('report_date', '')or ''
    
    font_style_column1 = 'Times New Roman'
    font_size_column1 = Pt(12)
    bold_column1 = True
    font_style_column2 = 'Times New Roman'
    font_size_column2 = Pt(12)
    bold_column2 = False
    for row in table.rows:
        row.cells[0].paragraphs[0].runs[0].font.name = font_style_column1
        row.cells[0].paragraphs[0].runs[0].font.size = font_size_column1
        row.cells[0].paragraphs[0].runs[0].bold = bold_column1
    for row in table.rows:
        row.cells[1].paragraphs[0].runs[0].font.name = font_style_column2
        row.cells[1].paragraphs[0].runs[0].font.size = font_size_column2
        row.cells[1].paragraphs[0].runs[0].bold = bold_column2

In [7]:
def patient_information(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(7.69)
    new_height = Cm(0.8)
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Patient_information.png', width=new_width, height=new_height)
    p = doc.add_paragraph()
    p.paragraph_format.line_spacing = 1
    p.paragraph_format.space_after = 12
    patient_information_table()

In [8]:
def oncoprecise_image(document):
    p = document.add_paragraph()
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    new_width = Cm(14.31)
    new_height = Cm(1.27)
    run = paragraph.add_run()
    run.add_picture('Oncoprecise.png', width=new_width, height=new_height)

In [9]:
def clinical_history(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(6.73)
    new_height = Cm(0.63)
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Clinical_history.png', width=new_width, height=new_height)
    # Data from Unipath file
    clinical_details_text = data.get("Clinical Details", "")
    clinical_details_text = clinical_details_text.replace("\n", "")
    p = document.add_paragraph()
    p.add_run(clinical_details_text)
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'

In [10]:
# Data from Unipath file
result_summary_str = data.get('Result_Summary', '')
result_summary = json.loads(result_summary_str)

# Extract TMB information
tmb_status = ""
tmb_score = ""
for entry in result_summary:
    if "Tumor mutational burden (TMB)- Status" in entry["Alteration Description "]:
        tmb_status = entry["Findings "].strip()
    elif "TMB Score (Mutation/mb)" in entry["Alteration Description "]:
        tmb_score = entry["Findings "].strip()
combined_tmb_value = f"{tmb_status} ({tmb_score} mut/mb)" if tmb_score else tmb_status

# Extract MSI information
msi_status = ""
msi_score = ""
for entry in result_summary:
    if "Microsatellite instability (MSI) -Status" in entry["Alteration Description "]:
        msi_status = entry["Findings "].strip()
    elif "MSI Score" in entry["Alteration Description "]:
        msi_score = entry["Findings "].strip()
combined_msi_value = f"{msi_status} (Score:{msi_score})" if msi_score else msi_status

# Extract LOH information
loh = ""
for entry in result_summary:
    if "Loss of Heterozygosity (LOH)" in entry["Alteration Description "]:
        loh = entry["Findings "].strip()
loh_score = f"LOH({loh})"

# Extract CNVs information
cnvs_findings = ""
for entry in result_summary:
    if "Copy number variants (CNVs)" in entry["Alteration Description "]:
        cnvs_findings = entry["Findings "].strip()

# Extract SNV identifiers and create DataFrame
snv_identifiers_list = data.get("SNV_Identifiers", "")
data_list = json.loads(snv_identifiers_list)
snv_df = pd.DataFrame(data_list)
snv_df.columns = snv_df.columns.str.strip()
snv_df['gene_mutation_list'] = snv_df.apply(lambda row: f"{row['Gene/Transcript'].split()[0]} ({row['Variant/Amino Acid Change'].split()[0]})", axis=1)

# Extract Fusion identifiers and create DataFrame
fusion_identifiers_list = data.get("Fusion_Identifiers", "")
data_list = json.loads(fusion_identifiers_list)
fusion_df = pd.DataFrame(data_list)
fusion_df.columns = fusion_df.columns.str.strip()

# Consolidate the results into a dictionary
table_data = {
    "Gene Mutations": '\n'.join(snv_df['gene_mutation_list']),
    "Gene Fusions": '\n'.join(fusion_df['Gene/Transcript']),
    "Copy number variants (CNVs)": cnvs_findings,
    "Loss of Heterozygosity (LOH)": loh_score,
    "Tumor mutational burden (TMB)": combined_tmb_value,
    "Microsatellite instability (MSI)": combined_msi_value,}

def report_summary_table(document, data):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0.14)
        section.right_margin = Cm(2.5)
    table = document.add_table(rows=1, cols=2)
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(7.51)
    table.columns[1].width = Cm(9.97)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style = 'Table Grid'
    first_row = table.rows[0].cells
    first_row[0].text = "Alteration Description"
    first_row[1].text = "Findings"
    for cell in first_row:
        cell.paragraphs[0].runs[0].bold = True
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        cell.paragraphs[0].runs[0].font.color.rgb = RGBColor(0, 0, 128) 
    row_height_cm = 1.04
    first_row = table.rows[0]
    first_row.height = Pt(row_height_cm * 28.3465)
    for key, value in data.items():
        row_cells = table.add_row().cells
        cell = row_cells[0]
        cell.text = key
        cell.paragraphs[0].runs[0].bold = True
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.size = Pt(12)
        row_cells[1].text = value
        cell1 = row_cells[1]
        cell1.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell1.paragraphs[0].runs[0].font.size = Pt(12)
    return table

In [11]:
def report_summary(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(5.82)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    run = paragraph.add_run()
    run.add_picture('Report_summary.png', width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2 * 0.42)
    
    FFPE=patient_info.get('FFPE_block_number', '')or '___' 
    tumor_cell=patient_info.get('tumor_cells', '')or '___'
    p.add_run("The test was performed on Block ")
    p.add_run(FFPE).bold = True
    p.add_run(" which showed presence of ")
    p.add_run(tumor_cell).bold = True
    p.add_run(" tumor cells.")    
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'

    report_summary_table(document, table_data)
    document.paragraphs[-1].paragraph_format.space_before = Pt(0)
    note1 = document.add_paragraph("*Note: This High TMB Score might be affected by higher deamination in this sample which may be due to the intrinsic nature of FFPE tissue.")
    note1.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note1.paragraph_format.left_indent = Cm(2.5)
    note1.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    note1.paragraph_format.line_spacing = 1
    for run in note1.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    note2 = document.add_paragraph("**Note: Assessment of HRR/HRD using orthogonal methods is recommended.")
    note2.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    note2.paragraph_format.left_indent = Cm(2.5)
    note2.paragraph_format.right_indent = Cm(2.5)
    note1.paragraph_format.space_after = Cm(0)
    for run in note2.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'

In [12]:
def therapeutic_summary(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(12.95)
    new_height = Cm(0.57)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Therapeutic_summary.png', width=new_width, height=new_height)
    line1_paragraph = document.add_paragraph("Therapeutic summary is relevant for the patient’s diagnosis")
    line1_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line1_paragraph.paragraph_format.left_indent = Cm(2.5)
    line1_paragraph.paragraph_format.right_indent = Cm(2.5)
    line1_paragraph.paragraph_format.space_after = Cm(0)
    line2_paragraph = document.add_paragraph("(Additional in-trial therapies relevant for this patient are listed in the 'clinical trials' section)")
    line2_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    line2_paragraph.paragraph_format.left_indent = Cm(2.5)
    line2_paragraph.paragraph_format.right_indent = Cm(2.5)
    line2_paragraph.paragraph_format.space_after = Cm(0)
    for run in line1_paragraph.runs + line2_paragraph.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'

In [13]:
def add_formatted_paragraph(doc, text):
    paragraph = doc.add_paragraph()
    run = paragraph.add_run(text)
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)
    paragraph.paragraph_format.line_spacing = 1.0
    paragraph.paragraph_format.left_indent = Cm(2.5)
    paragraph.paragraph_format.right_indent = Cm(2.5)
    return paragraph
def immunotherapy_markers(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(10.27)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Immunotherapy_markers.png', width=new_width, height=new_height)
    # Extract TMB information
    tmb_status = ""
    tmb_score = ""
    for entry in result_summary:
        if "Tumor mutational burden (TMB)- Status" in entry["Alteration Description "]:
            tmb_status = entry["Findings "].strip()
        elif "TMB Score (Mutation/mb)" in entry["Alteration Description "]:
            tmb_score = entry["Findings "].strip()
    combined_tmb_value = f"{tmb_status} ({tmb_score} mut/mb)" if tmb_score else tmb_status

    # Extract MSI information
    msi_status = ""
    msi_score = ""
    for entry in result_summary:
        if "Microsatellite instability (MSI) -Status" in entry["Alteration Description "]:
            msi_status = entry["Findings "].strip()
        elif "MSI Score" in entry["Alteration Description "]:
            msi_score = entry["Findings "].strip()
    combined_msi_value = f"{msi_status} (Score:{msi_score})" if msi_score else msi_status
    pdl1_status=patient_info.get('PD-L1_status', '')or ''
    pdl1_status_score=patient_info.get('PD-L1_status_score', '')or ''
    mrr_status =patient_info.get('MMR_status', '')or ''
    combined_pdl1_value = f"{pdl1_status}({pdl1_status_score})" if pdl1_status_score else pdl1_status
    add_formatted_paragraph(document, f"1- PDL1 by IHC (SP263)- {combined_pdl1_value}")
    add_formatted_paragraph(document, f"2- Tumor Mutation Burden- {combined_tmb_value}")
    add_formatted_paragraph(document, f"3- Microsatellite Instability- {combined_msi_value}")
    add_formatted_paragraph(document, f"4- MMR (Mismatch repair) status- {mrr_status}")

In [14]:
def add_hyperlink(paragraph, text, url):
    part = paragraph.part
    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
    hyperlink = docx.oxml.shared.OxmlElement('w:hyperlink')
    hyperlink.set(docx.oxml.shared.qn('r:id'), r_id)
    new_run = docx.text.run.Run(docx.oxml.shared.OxmlElement('w:r'), paragraph)
    new_run.text = text
    new_run.style = get_or_create_hyperlink_style(part.document)
    hyperlink.append(new_run._element)
    paragraph._p.append(hyperlink)
    return hyperlink

def get_or_create_hyperlink_style(document):
    if "Hyperlink" not in document.styles:
        if "Default Character Font" not in document.styles:
            default_character_font = document.styles.add_style("Default Character Font", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
            default_character_font.element.set(docx.oxml.shared.qn('w:default'), "1")
            default_character_font.priority = 1
            default_character_font.hidden = True
            default_character_font.unhide_when_used = True
            del default_character_font
        hyperlink_style = document.styles.add_style("Hyperlink", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
        hyperlink_style.base_style = document.styles["Default Character Font"]
        hyperlink_style.unhide_when_used = True
        hyperlink_style.font.color.rgb = docx.shared.RGBColor(0x05, 0x63, 0xC1)
        hyperlink_style.font.underline = True
        del hyperlink_style
    return "Hyperlink"

def approved_immunotherapy(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
        
    new_width = Cm(9.58)
    new_height = Cm(0.69)
    
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Approved_immunotherapy.png', width=new_width, height=new_height)
    
    tumor_types = patient_info.get('Main Tumor Type', '').split(', ')
    #print(tumor_types)
    fda_approved_df = pd.read_excel('running FDA-approved cancer Immunotherapies.xlsx')
    
    for tumor_type in tumor_types:
        matching_rows2 = fda_approved_df[fda_approved_df['Main tumor type'].str.contains(fr'{tumor_type}', case=False, regex=True, na=False, flags=re.IGNORECASE)]        
        for index, row in matching_rows2.iterrows():
            drug_name = row['PD-1/PD-L1 inhibitor']
            link_holder = document.add_paragraph()
            link_holder.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            link_holder.paragraph_format.left_indent = Cm(2.5)
            link_holder.paragraph_format.right_indent = Cm(2.5)
            drug=drug_name.replace(" ","+")
            add_hyperlink(link_holder, drug_name, f"https://www.google.com/search?q={drug}+mechanism+of+action")

            main_tumor_type=row['Main tumor type']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(main_tumor_type)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'
            detailed_tumor_type=row['FDA-approved indications']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(detailed_tumor_type)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'
            details_of_approval = row['Details of approval']
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(details_of_approval)
            for run in p.runs:
                run.font.size = Pt(12)
                run.font.name ='Times New Roman'

In [15]:
def intrial_immunotherapy(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(9.58)
    new_height = Cm(0.56)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Intrial_immunotherapy.png', width=new_width, height=new_height)
    paragraph = document.add_paragraph("Multiple immunotherapies are being investigated in clinical trials for this patient’s genomic profile and/or diagnosis (Please see “Clinical trials” section for more details).")
    paragraph.paragraph_format.left_indent = Cm(2.5)
    paragraph.paragraph_format.right_indent = Cm(2.5)
    run_text = paragraph.runs[0]
    font = run_text.font
    font.name = 'Times New Roman'
    font.size = Pt(12)

In [16]:
# Data from Unipath file
snv_identifiers_list = data.get("SNV_Identifiers", "")
data_list = json.loads(snv_identifiers_list)
snv_df = pd.DataFrame(data_list)
snv_df.rename(columns={'Gene/Transcript ': 'Gene'}, inplace=True)
snv_df = snv_df.drop(['Locus '], axis=1)
snv_df.rename(columns={'Variant/Amino Acid Change ': 'Genomic alteration'}, inplace=True)
snv_df.rename(columns={'Variant classification ': 'Clinical Significance'}, inplace=True)
snv_df.rename(columns={'TIER classification ': 'AMP^ Classification'}, inplace=True)
snv_df.rename(columns={'Total Coverage/VAF ': 'Variant Allelic Frequency (VAF%)'}, inplace=True)
snv_df['Variant Allelic Frequency (VAF%)'] = snv_df['Variant Allelic Frequency (VAF%)'].str.replace(r'\d+X\s*', '', regex=True)
fusion_identifiers_list = data.get("Fusion_Identifiers", "")
fusion_data_list = json.loads(fusion_identifiers_list)
fusion_df = pd.DataFrame(fusion_data_list)
fusion_df.rename(columns={'Gene/Transcript ': 'Gene'}, inplace=True)
# Table content and styling 
def variant_detail_table1(document, data):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.5)
        section.right_margin = Cm(2)
    if not snv_df.empty:
        table = document.add_table(rows=snv_df.shape[0] + 1, cols=snv_df.shape[1])
        table.style = 'Table Grid'
        table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for col_num, col_name in enumerate(snv_df.columns):
            cell = table.cell(0, col_num)
            cell.text = col_name
            run = cell.paragraphs[0].runs[0]
            run.bold = True
            run.font.name = 'Times New Roman'
            run.font.size = Pt(10)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for row_num in range(snv_df.shape[0]):
            for col_num, value in enumerate(snv_df.iloc[row_num]):
                cell = table.cell(row_num + 1, col_num)
                cell.text = str(value)
                run = cell.paragraphs[0].runs[0]
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)
                cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        if snv_df.shape[0] < len(table.rows):
            last_row = table.rows[-1]
            if all(cell.text == '' for cell in last_row.cells):
                table._element.remove(last_row._element)

    if not fusion_df.empty:
        table = document.add_table(rows=fusion_df.shape[0] + 1, cols=fusion_df.shape[1])
        table.style = 'Table Grid'
        table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        for col_num, col_name in enumerate(fusion_df.columns):
            cell = table.cell(0, col_num)
            cell.text = col_name
            run = cell.paragraphs[0].runs[0]
            run.bold = True
            run.font.name = 'Times New Roman'
            run.font.size = Pt(10)
            cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        for row_num in range(min(snv_df.shape[0], fusion_df.shape[0])):
            for col_num, value in enumerate(fusion_df.iloc[row_num]):
                cell = table.cell(row_num + 1, col_num)
                cell.text = str(value)
                run = cell.paragraphs[0].runs[0]
                run.font.name = 'Times New Roman'
                run.font.size = Pt(10)
                cell.paragraphs[0].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        if fusion_df.shape[0] < len(table.rows):
            last_row = table.rows[-1]
            if all(cell.text == '' for cell in last_row.cells):
                table._element.remove(last_row._element)
    return table


In [17]:
snv_identifiers_list = data.get("SNV_Identifiers", "")
def variant_detail_table2(document, variants):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.5)
        section.right_margin = Cm(2)
    
    table = document.add_table(rows=4, cols=9)  
    table.autofit = False
    table.style = 'Table Grid'
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    
    widths = [2, 1.5, 1.5, 2, 1.5, 1.5, 1.5, 1.5, 1.5]  
    for i, width in enumerate(widths):
        table.cell(0, i).width = Cm(width)

    heights = [1, 1, 1, 1]
    for i, height in enumerate(heights):
        table.rows[i].height = Cm(height)
        
    header = table.rows[0].cells
    header[0].text = "Variant"
    table.cell(0, 0).merge(table.cell(1, 0))
    header[1].text = "dbSNP\nrs ID"
    table.cell(0, 1).merge(table.cell(1, 1))
    header[2].text = "COSMIC"
    table.cell(0, 2).merge(table.cell(1, 2))
    header[3].text = "ClinVar"
    table.cell(0, 3).merge(table.cell(1, 3))
    header[4].text = "In-silico predictors"
    table.cell(0, 4).merge(table.cell(0, 6))
    header[7].text = "Population Database"
    table.cell(0, 7).merge(table.cell(0, 8))
    
    second_row = table.rows[1].cells
    second_row[0].text = "Variant"
    second_row[1].text = "dbSNP\nrs ID"
    second_row[2].text = "COSMIC"
    second_row[3].text = "ClinVar"
    second_row[4].text = "SIFT"
    second_row[5].text = "Polyphen"
    second_row[6].text = "MT2"
    second_row[7].text = "gnomAD"
    second_row[8].text = "ExAc"
    
    header = table.rows[0].cells
    second_row = table.rows[1].cells
    for i in range(min(len(header), len(second_row), 9)):
        header[i].paragraphs[0].runs[0].font.bold = True
        header[i].paragraphs[0].runs[0].font.name = 'Times New Roman'
        header[i].paragraphs[0].runs[0].font.size = Pt(11)
        header[i].paragraphs[0].paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        header[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER

        second_row[i].paragraphs[0].runs[0].font.bold = True
        second_row[i].paragraphs[0].runs[0].font.name = 'Times New Roman'
        second_row[i].paragraphs[0].runs[0].font.size = Pt(11)
        second_row[i].paragraphs[0].paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        second_row[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
        
    data_list = json.loads(snv_identifiers_list)
    total_rows_needed = max(len(data_list), len(variants)) + 2  
    while len(table.rows) < total_rows_needed:
        table.add_row()
    for i, data in enumerate(data_list):
        row = table.rows[i + 2]
        row.cells[0].text = data.get("Gene/Transcript ", "")
        
    for i, variant in enumerate(variants):
        row = table.rows[i + 2]
        row.cells[1].text = variant.get("rs_id", "")
        row.cells[2].text = variant.get("cosmic_id", "")
        row.cells[3].text = variant.get("clinvar_id", "")
        row.cells[4].text = variant["predictors"].get("SIFT", "")
        row.cells[5].text = variant["predictors"].get("Polyphen", "")
        row.cells[6].text = variant["predictors"].get("MT2", "")
        row.cells[7].text = variant['population_database'].get('gnomAD', "")
        row.cells[8].text = variant['population_database'].get('ExAc', "")

    for row in table.rows[2:]:
        for cell in row.cells:
            cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            for paragraph in cell.paragraphs:
                if paragraph.runs and paragraph.runs[0].text.strip():
                    paragraph.runs[0].font.size = Pt(10)
                    paragraph.runs[0].font.name = 'Times New Roman'
                    paragraph.paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

    return table

In [18]:
def variant_details(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(9.58)
    new_height = Cm(0.56)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Variant_details.png', width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Table-1: List of variants identified in this sample").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    variant_detail_table1(document, fusion_df)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Refer to supplementary information for the classification criteria details. ^")
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Table-2: Significance of Variants reported in database").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    variants = [ { "rs_id": "rs1131691026",
        "cosmic_id": "COSM10727",
        "clinvar_id": "VCV000428890",
        "predictors": {"SIFT": "-", "Polyphen": "-",  "MT2": "D",},
        "population_database": { "gnomAD": "-", "ExAc": "-",}},
      {"rs_id": "rs756438082",
        "cosmic_id": "-",
        "clinvar_id":"VCV000526349",
        "predictors": {"SIFT": "D", "Polyphen": "D", "MT2": "D", },
        "population_database": {"gnomAD": "0.01%", "ExAc": "0.01%", }}]
    variant_detail_table2(document, variants)
    
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("P- Pathogenic, D- Damaging, B-Benign")
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()

In [19]:
amino_acid_mapping = {
    'Ala': 'A',
    'Arg': 'R',
    'Asn': 'N',
    'Asp': 'D',
    'Cys': 'C',
    'Glu': 'E',
    'Gln': 'Q',
    'Gly': 'G',
    'His': 'H',
    'Ile': 'I',
    'Leu': 'L',
    'Lys': 'K',
    'Met': 'M',
    'Phe': 'F',
    'Pro': 'P',
    'Ser': 'S',
    'Ter': '*',
    'Thr': 'T',
    'Trp': 'W',
    'Tyr': 'Y',
    'Val': 'V'
}

def convert_variation(variation):
    if 'del' in variation and 'ins' in variation and 'fs' in variation:
        gene, rest = variation.split(':c.')
        match = re.match(r"(\d+)_(\d+)del([A-Z]+)ins([A-Z]+)\:p\.([A-Za-z0-9]+)(\d+)([A-Za-z]+|\*|fs\*\d*)?", rest)
    elif 'ins' in variation and 'fs' in variation:
        gene, rest = variation.split(':c.')
        match = re.match(r"(\d+)_(\d+)ins([A-Z0-9]*)\:p\.([A-Za-z0-9]+)(\d+)([A-Za-z]+|\*|fs\*\d*)?", rest)
    elif 'del' in variation:
        gene, rest = variation.split(':c.')
        match = re.match(r"(\d+)del([A-Z]*)\:p\.([A-Za-z0-9]+)(\d+)([A-Za-z]+|\*|fs\*\d*)?", rest)
    else:
        match = re.match(r"([A-Z0-9]+):c\.(\d+)([A-Z]*)>([A-Z]*)\:p\.([A-Za-z0-9]+)(\d+)([A-Za-z]+|\*|fs\*\d*)?", variation)
    if match:
        if 'del' in variation and 'ins' in variation and 'fs' in variation:
            position_start, position_end, ref_base, alt_base, aa_change, aa_position, aa_variant = match.groups()
            aa_change_one_letter = amino_acid_mapping.get(aa_change, aa_change)
            aa_variant_one_letter = ''.join(amino_acid_mapping.get(aa, aa) for aa in (aa_variant or ''))
            result = f"{gene} {aa_change_one_letter}{aa_position}{aa_variant_one_letter}"
            result_modified = f"{gene} {aa_change_one_letter}{aa_position}{aa_variant_one_letter.replace('Ter', '*')}"
        elif 'ins' in variation and 'fs' in variation:
            position, alt_base, aa_change, aa_position, aa_variant, _ = match.groups()
            aa_change_one_letter = amino_acid_mapping.get(aa_change, aa_change)
            aa_variant_one_letter = ''.join(amino_acid_mapping.get(aa, aa) for aa in (aa_variant or ''))
            aa_variant_one_letter = aa_variant_one_letter.replace('ins', '').replace('fs', 'Ter').replace('TerTer', 'Ter')
            aa_variant_one_letter = aa_variant_one_letter.split(':')[-1].split('Ter')[0]
            result = f"{gene.split(':')[0]} {aa_position}{aa_variant_one_letter}"
            result_modified = f"{gene.split(':')[0]} {aa_position}{aa_variant_one_letter.replace('Ter', '*')}"
        elif 'del' in variation and 'fs' in variation:
            position, ref_base, aa_change, aa_position, aa_variant = match.groups()
            aa_change_one_letter = amino_acid_mapping.get(aa_change, aa_change)
            aa_variant_one_letter = ''.join(amino_acid_mapping.get(aa, aa) for aa in (aa_variant or ''))
            result = f"{gene} {aa_change_one_letter}{aa_position}{aa_variant_one_letter}"
            result_modified = f"{gene} {aa_change_one_letter}{aa_position}{aa_variant_one_letter.replace('Ter', '*')}"
        elif 'del' in variation and variation.endswith('del'):
            position, aa_change, aa_position, aa_variant = match.groups()[0], match.groups()[2], match.groups()[3], match.groups()[4]
            aa_change_one_letter = amino_acid_mapping.get(aa_change, aa_change)
            aa_variant_one_letter = ''.join(amino_acid_mapping.get(aa, aa) for aa in (aa_variant or ''))
            result = f"{gene} {aa_change_one_letter}{aa_position}{aa_variant_one_letter}"
            result_modified = f"{gene} {aa_change_one_letter}{aa_position}{aa_variant_one_letter.replace('Ter', '*')}"
        else:
            gene, position, ref_base, alt_base, aa_change, aa_position, aa_variant = match.groups()
            aa_change_one_letter = ''.join(amino_acid_mapping.get(aa, aa) for aa in (aa_change or ''))
            aa_variant_one_letter = ''.join(amino_acid_mapping.get(aa, aa) for aa in (aa_variant or ''))
            result = f"{gene} {aa_change_one_letter}{aa_position}{aa_variant_one_letter}"
            for aa, replacement in amino_acid_mapping.items():
                aa_change = aa_change.replace(aa, replacement) if aa_change else ''
                aa_variant = aa_variant.replace(aa, replacement) if aa_variant else ''
            result_modified = f"{gene} {aa_change}{aa_position}{aa_variant.replace('Ter', '*')}"
        for aa, replacement in amino_acid_mapping.items():
            result_modified = result_modified.replace(aa, replacement)
        return result_modified
    elif re.match(r"([A-Z0-9]+)\(\d+\) - ([A-Z0-9]+)\(\d+\)", variation):
        gene1, count1, gene2, count2 = re.findall(r"([A-Z0-9]+)\((\d+)\) - ([A-Z0-9]+)\((\d+)\)", variation)[0]
        gene_names_sorted = sorted([gene1, gene2])
        result = f"{gene_names_sorted[0]} - {gene_names_sorted[1]}"
        return result
    else:
        result = f"{variation} Amplification"
        return result

In [20]:
def add_hyperlink(paragraph, drug_name, url):
    part = paragraph.part
    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
    hyperlink = docx.oxml.shared.OxmlElement('w:hyperlink')
    hyperlink.set(docx.oxml.shared.qn('r:id'), r_id)
    new_run = docx.text.run.Run(docx.oxml.shared.OxmlElement('w:r'), paragraph)
    new_run.text = drug_name
    new_run.style = get_or_create_hyperlink_style(part.document)
    hyperlink.append(new_run._element)
    paragraph._p.append(hyperlink)
    return hyperlink

def get_or_create_hyperlink_style(document):
    if "Hyperlink" not in document.styles:
        if "Default Character Font" not in document.styles:
            default_character_font = document.styles.add_style("Default Character Font", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
            default_character_font.element.set(docx.oxml.shared.qn('w:default'), "1")
            default_character_font.priority = 1
            default_character_font.hidden = True
            default_character_font.unhide_when_used = True
            del default_character_font
        hyperlink_style = document.styles.add_style("Hyperlink", docx.enum.style.WD_STYLE_TYPE.CHARACTER, True)
        hyperlink_style.base_style = document.styles["Default Character Font"]
        hyperlink_style.unhide_when_used = True
        hyperlink_style.font.color.rgb = docx.shared.RGBColor(0x05, 0x63, 0xC1)
        hyperlink_style.font.underline = True
        del hyperlink_style
    return "Hyperlink"

def extract_hyperlink_data(drug_des):
    pattern = re.compile(r'Drug: (\w+)', re.IGNORECASE)
    matches = re.findall(pattern, drug_des)
    return matches

def set_run_color(run, color):
    font = run.font
    font.color.rgb = color
    
def generate_gene_variations(formatted_gene_name):
    gene_variations = []
    gene_variations.append(formatted_gene_name)
    gene_names = [name.strip() for name in formatted_gene_name.split('-')]
    gene_variations.append(f'{gene_names[1]} - {gene_names[0]}')
    gene_variations.append(f'{gene_names[0]} Fusion')
    gene_variations.append(f'{gene_names[1]} Fusion')
    return gene_variations

def add_gene_summary_text(document, gene_name, ncbi_summary, gene_role, gene_ckb_des, my_cancer_des, prognosis_pubmed_des, prognosis_germline_des, approved_therapy_des, drug_des, is_fusion=False):
    #Gene Heading
    p = document.add_paragraph()
    p_heading = f"{gene_name} {'Fusion' if is_fusion else ''}\n"
    p.add_run(p_heading).bold = True
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    for run in p.runs:
        run.font.name = 'Times New Roman'
        run.font.size = Pt(12) 
    formatted_gene_name = gene_name.replace("\n", "").replace(" - ", "-").replace("-", " - ")
    formatted_gene_name = formatted_gene_name.replace("c.", ":c.").replace("p.", ":p.")
    formatted_gene_name = formatted_gene_name.replace(" ", "")
    formatted_gene_name = formatted_gene_name.replace(" -  - ", " - ")
    formatted_gene_name = formatted_gene_name.replace("--", " - ")
    formatted_gene_name = formatted_gene_name.replace("-:", "-")

    #Gene Summary
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Gene Summary").bold = True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(ncbi_summary):
        ncbi_summary = 'There is no NCBI summary information for this gene'
    ncbi_summary = re.sub(r'\[provided by RefSeq, [A-Za-z]+ \d+\]', '', ncbi_summary)
    p.add_run(ncbi_summary)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
        
    #Altered variant
    gene_variants_data = pd.read_excel('alter_variant_data.xlsx')
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run("Altered variant").bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    gene_variants_data = gene_variants_data.groupby('Gene_name')['Variant'].agg(lambda x: '\n'.join(x)).reset_index()
    matching_gene = gene_variants_data[gene_variants_data['Gene_name'] == formatted_gene_name]
    
    if not matching_gene.empty:
        variants = matching_gene['Variant'].tolist()
        for variant in variants:
            variant = variant.replace('\n', ' ')
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(variant)
            for run in p.runs:
                run.font.size = Pt(11)
                run.font.name = 'Times New Roman'
            if '-' in formatted_gene_name:
                result_modified = convert_variation(formatted_gene_name)
                gene_variations_data = generate_gene_variations(result_modified)
                p = document.add_paragraph('Link(s) to OncoKB for the variant:\n')
                p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
                p.paragraph_format.left_indent = Cm(2.5)
                p.paragraph_format.right_indent = Cm(2.5)
                for gene_variant in gene_variations_data:
                    hyperlink_text = re.sub(r'[\s-]+', '/', gene_variant)
                    hyperlink_url = f'https://www.oncokb.org/gene/{hyperlink_text}'
                    add_hyperlink(p, gene_variant, hyperlink_url)
                    p.add_run('\n')
                for run in p.runs:
                    run.font.size = Pt(11)
                    run.font.name = 'Times New Roman'
            else: 
                result_modified = convert_variation(formatted_gene_name)
                p = document.add_paragraph('Link(s) to OncoKB for the variant:\n')
                p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
                p.paragraph_format.left_indent = Cm(2.5)
                p.paragraph_format.right_indent = Cm(2.5)
                hyperlink_text = result_modified.replace('-', '/').replace(' ', '/')
                hyperlink_url = f'https://www.oncokb.org/gene/{hyperlink_text}'
                add_hyperlink(p, result_modified, hyperlink_url)
                for run in p.runs:
                    run.font.size = Pt(11)
                    run.font.name = 'Times New Roman'
    else:
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run("No variant information available for this gene.")
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
        
    # Gene disease association
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Gene -Disease association').bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(gene_role):
        gene_role = 'There is no Gene role information for this gene'
    p.add_run(gene_role)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(gene_ckb_des):
        gene_ckb_des = 'There is no CKB core information for this gene'
    p.add_run(gene_ckb_des)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    if pd.isna(my_cancer_des):
        my_cancer_des = 'There is no My cancer genome information for this gene'
    else:
        my_cancer_des = str(my_cancer_des)
    p.add_run(my_cancer_des)
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
        
    #Prognostic significance
    if prognosis_pubmed_des is not None:
        prognosis_pubmed_des = str(prognosis_pubmed_des)
        pattern = re.compile(r'\(PMID: \d+\)\.')
        prognosis_pubmed_des = re.sub(pattern, lambda x: x.group() + '\n', prognosis_pubmed_des)

        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('Prognostic significance').bold=True
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name ='Times New Roman'
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        if prognosis_pubmed_des.lower() == 'nan':
            prognosis_pubmed_des = 'There is no Prognostic significance information for this gene'
        p.add_run(prognosis_pubmed_des)
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
    if prognosis_germline_des is not None:
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run('Prognostic significance of germline mutation').bold=True
        for run in p.runs:
            run.font.size = Pt(12)
            run.font.name ='Times New Roman'
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        if prognosis_germline_des.lower() == 'nan':
            prognosis_germline_des = 'There is no Prognostic significance of germline mutation information for this gene in ACMG Clinvar data'
        p.add_run(prognosis_germline_des)
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
            
    #Therapy
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Therapy' + '\n' + 'Approved therapy').bold = True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    lines = approved_therapy_des.split('\n')
    color_mapping = {
    'MediumPurple': RGBColor(147, 112, 219),
    'Pink': RGBColor(255, 20, 147),
    'Green': RGBColor(0, 128, 0),
    'Red': RGBColor(255, 0, 0),
    'Black': RGBColor(0, 0, 0)}
    current_color = None
    for line in lines:
        p = document.add_paragraph()
        p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        p.paragraph_format.left_indent = Cm(2.5)
        p.paragraph_format.right_indent = Cm(2.5)
        p.add_run(line)
        if line.startswith('Biomarker Criteria:'):
            current_color = color_mapping['Green']
        elif line.startswith('Predicted Response:'):
            current_color = color_mapping['Red']
        elif line.startswith('Clinical Setting(s):'):
            current_color = color_mapping['Black']
        elif line.startswith('Note:'):
            current_color = color_mapping['Black']
        elif line.startswith('Drug(s)'):
            current_color = color_mapping['MediumPurple']
        elif line.startswith('Cancer:'):
            current_color = color_mapping['Pink']
        for run in p.runs:
            run.font.size = Pt(11)
            run.font.name = 'Times New Roman'
            set_run_color(run, current_color)
        
    #In trial therapy
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('In trial therapy').bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
        
    #Drug resistance information
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Drug resistance information').bold=True
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    if pd.isna(drug_des):
        drug_des = 'There is no drug resistance information for this gene'
    p.add_run('\n')
    for line in drug_des.split('\n'):
        if line.startswith('Drug:'):
            drug_name = line.split('Drug:')[1].strip()
            drug = drug_name.replace(" ","+")
            hyperlink_run = add_hyperlink(p, drug_name, f"https://www.google.com/search?q={drug}+mechanism+of+action")
            p.add_run(line.split(drug_name)[1])
        else:
            p = document.add_paragraph()
            p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
            p.paragraph_format.left_indent = Cm(2.5)
            p.paragraph_format.right_indent = Cm(2.5)
            p.add_run(line)
            for run in p.runs:
                run.font.size = Pt(11)
                run.font.name ='Times New Roman'

In [21]:
def extract_variant_info(genes_involved, variants_description):
    gene_variants_data = {'Gene_name': pd.Series([], dtype='object'), 'Variant': pd.Series([], dtype='object')}
    current_gene = ""

    for variant in variants_description:
        for gene in genes_involved:
            if gene in variant and variant.startswith(gene):
                current_gene = gene
                break
        if current_gene:
            gene_variants_data['Gene_name'] = gene_variants_data['Gene_name'].append(pd.Series([current_gene], dtype='object'), ignore_index=True)
            gene_variants_data['Variant'] = gene_variants_data['Variant'].append(pd.Series([variant], dtype='object'), ignore_index=True)
        elif not gene_variants_data.empty:
            gene_variants_data['Variant'].iloc[-1] += "\n" + variant
            
    gene_variants_data_df = pd.DataFrame(gene_variants_data)
    gene_variants_data_df.to_excel('alter_variant_data.xlsx', index=False)

In [22]:
def gene_varient_description(document, genes_involved):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(9.81)
    new_height = Cm(0.56)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Gene_variant_des.png', width=new_width, height=new_height)
    snv_identifiers_list = data.get("SNV_Identifiers", "")
    data_list = json.loads(snv_identifiers_list)
    snv_df1 = pd.DataFrame(data_list)
    snv_df1.columns = snv_df1.columns.str.strip()
    snv_df1['combined'] = snv_df1.apply(lambda row: f"{row['Gene/Transcript'].split()[0]}\n{row['Variant/Amino Acid Change']}", axis=1)
    snv_df1['Gene_Name'] = snv_df1.apply(lambda row: f"{row['Gene/Transcript'].split()[0]}", axis=1)
    snv_df1.rename(columns={'Total Coverage/VAF ': 'Variant Allelic Frequency (VAF%)'}, inplace=True)
    snv_df1['Variant Allelic Frequency (VAF%)'] = snv_df['Variant Allelic Frequency (VAF%)'].str.replace(r'\d+X\s*', '', regex=True)
    ncbi_df = pd.read_excel('19K_Gene_GC_NCBI_Info.xlsx')
    ncbi_df.drop(['GC_Name', 'GC_Aliases', 'Gene_ID', 'NCBI_Official_Symbol', 'NCBI_Full_Name', 'NCBI_Aliases'], axis=1, inplace=True)
    snv_df1 = pd.merge(snv_df1, ncbi_df, on='Gene_Name', how='inner')
    ckbcore_df = pd.read_excel('Running Copy of Gene description_MyCancerGenome_CkBcore.xlsx')
    ckbcore_df.rename(columns={'Gene name': 'Gene_Name'}, inplace=True)
    snv_df1 = pd.merge(snv_df1, ckbcore_df, on='Gene_Name', how='inner')
    prognosis_df = pd.read_excel('V2 Running Copy of Information related to prognosis.xlsx') 
    drug_res_df = pd.read_excel('V2 Running Copy of Drug resistance information.xlsx')
    tumor_types = [tumor.lower() for tumor in patient_info.get('Main Tumor Type', '').split(', ')]
    prognosis_df['Main Tumor type']=prognosis_df['Main Tumor type'].str.lower()
    prognosis_filtered_df = prognosis_df[(prognosis_df['Gene'].isin(snv_df1['Gene_Name'])) & (prognosis_df['Main Tumor type'].isin(tumor_types))]
    prognosis_filtered_df.rename(columns={'Gene': 'Gene_Name'}, inplace=True)
    prognosis_filtered_df =prognosis_filtered_df.groupby('Gene_Name')['Prognostic significance'].apply(lambda x: '\n'.join(x)).reset_index()
    drug_res_df['Main Tumor type'] = drug_res_df['Main Tumor type'].str.lower()
    drug_res_filtered_df = drug_res_df[(drug_res_df['Gene'].isin(snv_df1['Gene_Name'])) & (drug_res_df['Main Tumor type'].isin(tumor_types))]
    drug_res_filtered_df.rename(columns={'Gene': 'Gene_Name'}, inplace=True)
    drug_res_filtered_df["Drug_name_description"] = 'Drug: '+ drug_res_filtered_df["Potential drug \nresistance"].fillna('') + '\n' + drug_res_filtered_df["Description"].fillna('')+ '\n' 
    drug_res_filtered_df = drug_res_filtered_df.groupby('Gene_Name')['Drug_name_description'].apply(lambda x: '\n'.join(x)).reset_index()
    snv_df1 = pd.merge(snv_df1, prognosis_filtered_df, on='Gene_Name', how='outer')
    snv_df1 = pd.merge(snv_df1, drug_res_filtered_df, on='Gene_Name', how='outer')
    snv_df1.drop_duplicates(inplace=True)
    snv_df1.dropna(subset=["Gene/Transcript", "Locus", "Variant/Amino Acid Change"], inplace=True)
    variants_description = data.get("Variants Description", [])
    
    for index, row in snv_df1.iterrows():
        gene_name = row['combined']
        formatted_gene_name = gene_name.replace("\n", "").replace(" - ", "-").replace("-", " - ")
        formatted_gene_name = formatted_gene_name.replace("c.", ":c.").replace("p.", ":p.")
        formatted_gene_name = formatted_gene_name.replace(" ", "")
        formatted_gene_name = formatted_gene_name.replace(" -  - ", " - ")
        formatted_gene_name = formatted_gene_name.replace("--", " - ")
        formatted_gene_name = formatted_gene_name.replace("-:", "-")
        genes_involved.append(formatted_gene_name)
    for gene_pair in fusion_df['Gene'].unique():
        formatted_gene_name = gene_pair.replace("\n", "").replace(" - ", "-").replace("-", " - ")
        formatted_gene_name = formatted_gene_name.replace("c.", ":c.").replace("p.", ":p.")
        formatted_gene_name = formatted_gene_name.replace(" ", "")
        formatted_gene_name = formatted_gene_name.replace(" -  - ", " - ")
        formatted_gene_name = formatted_gene_name.replace("--", " - ")
        formatted_gene_name = formatted_gene_name.replace("-:", "-")
        genes_involved.append(formatted_gene_name)
    extract_variant_info(genes_involved, variants_description)
    prognosis_germline_des=''
    approved_therapy_des=''
    for index, row in snv_df1.iterrows():
        gene_name = row['combined']
        gene =row['Gene_Name']
        ncbi_summary = row['NCBI_Summary']
        gene_roles=row['Gene Role']
        gene_roles = gene_roles.split(' (')[0].replace('Both: ', '')
        gene_role = re.sub(r'\([^)]*\)', '', gene_roles)
        gene_ckb_des=row['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]']                                                                                                                                                                                        
        #my_cancer_des=row ['Description                                                                                                                                                                                                                Source: My Cancer Genome\nhttps://www.mycancergenome.org/content/biomarkers/                                                                                                                                        ']
        prognosis_des=row['Prognostic significance']
        variant_classification = row['Variant classification']
        vaf_percentage = row['Variant Allelic Frequency (VAF%)']
        vaf_percentage = vaf_percentage.replace('%', '').strip()
        if 'Pathogenic' in variant_classification or 'Likely Pathogenic' in variant_classification and float(vaf_percentage) >= 20:
            #print(gene)
            medgen_df = ACMG_dataextraction(gene)
            if not medgen_df.empty:
                matching_rows = medgen_df[medgen_df['Gene_Name'] == gene]
                if not matching_rows.empty and 'disease_characteristics' in matching_rows:
                    prognosis_germline_des = ' '.join(matching_rows['disease_characteristics'])
            else:
                prognosis_germline_des = 'There is no Prognostic significance of germline mutation information for this gene in ACMG Clinvar data'        
        df=pd.DataFrame()
        if ' ' in gene.strip():
            gene=gene.lower()
            gene=gene.replace(' ', '-')
            df=pd.concat([df,For_Alterations(gene)])
        elif ' ' not in gene.lower().strip():
            df=pd.concat([df,For_Genes(gene)])

        condition = (~df['Drugs'].eq('')) & (~df['Cancer'].eq('')) & (~df['Biomarker_Criteria'].eq(''))
        df['my_cancer_data'] = ''
        df.loc[condition, 'my_cancer_data'] = 'Drug(s): ' + df.loc[condition, 'Drugs'] + '\n' + 'Cancer: ' + df.loc[condition, 'Cancer'] + '\n' + df.loc[condition, 'Biomarker_Criteria']
        df = df[df['Gene_Name'] == gene]
        approved_therapy_des = '\n'.join(df['my_cancer_data']) if not df.empty and 'my_cancer_data' in df else 'MY CANCER GENOME COULD NOT BE EXTRACTED'       
        my_cancer_des = df['Gene_Role'].iloc[0].replace('\n', '').replace('  ', '') if not df.empty else 'There is no My cancer genome information for this gene'
        drug_des=row['Drug_name_description']
        add_gene_summary_text(document, gene_name, ncbi_summary, gene_role, gene_ckb_des, my_cancer_des, prognosis_des, prognosis_germline_des, approved_therapy_des, drug_des)

    prognosis_des='' 
    prognosis_germline_des =''
    approved_therapy_des=''
    for gene_pair in fusion_df['Gene'].unique():
        genes = re.findall(r'\b\w+\b', gene_pair)
        gene_summaries =[]
        gene_ckb_des =[]
        drug_des =[]
        for gene in genes:
            matching_rows = ncbi_df[ncbi_df['Gene_Name'].str.contains(fr'^\b{gene}\b$', case=False, regex=True)]
            for index, row in matching_rows.iterrows():
                gene_name = row['Gene_Name']
                ncbi_summary = row['NCBI_Summary']
                gene_summaries.append(gene_name)
                gene_summaries.append(ncbi_summary)
                gene_summaries.append("")
            matching_rows1 = ckbcore_df[ckbcore_df['Gene_Name'].str.contains(fr'^\b{gene}\b$', case=False, regex=True)]
            for index, row in matching_rows1.iterrows():
                gene_name = row['Gene_Name']
                gene_roles=row['Gene Role']
                gene_roles = gene_roles.split(' (')[0].replace('Both: ', '')
                gene_role = re.sub(r'\([^)]*\)', '', gene_roles)
                gene_des=row['Gene Description                                                                                                                                                                                        [Source : CKB core  https://ckb.jax.org/gene/grid]']                                                                                                                                                                                        
                #my_cancer_des=row ['Description                                                                                                                                                                                                                Source: My Cancer Genome\nhttps://www.mycancergenome.org/content/biomarkers/                                                                                                                                        ']
                df=pd.DataFrame()
                if ' ' in gene_name.strip():
                    gene=gene_name.lower()
                    gene=gene.replace(' ', '-')
                    df=pd.concat([df,For_Alterations(gene)])
                elif ' ' not in gene.lower().strip():
                    df=pd.concat([df,For_Genes(gene)])
                my_cancer_des = df['Gene_Role'].iloc[0].replace('\n', '').replace('  ', '') if not df.empty else 'There is no My cancer genome information for this gene'
                gene_ckb_des.append(gene_name)
                gene_ckb_des.append(gene_role)
                gene_ckb_des.append(gene_des)
                gene_ckb_des.append("")
                gene_ckb_des.append(my_cancer_des)
                gene_ckb_des.append("")
            matching_rows3 = prognosis_df[prognosis_df['Gene'].str.contains(fr'^\b{gene}\b$', case=False, regex=True) & (prognosis_df['Main Tumor type'].isin(tumor_types))]
            for index, row in matching_rows3.iterrows():
                prognosis_des=row['Prognostic significance']
            matching_row4 = drug_res_df[drug_res_df['Gene'].str.contains(fr'^\b{gene}\b$', case=False, regex=True) & (drug_res_df['Main Tumor type'].isin(tumor_types))]
            for index, row in matching_row4.iterrows():
                drug_name=row['Potential drug \nresistance']
                drug_name = f"Drug: {drug_name}"
                drug_description=row['Description']
                drug_des.append(drug_name)
                drug_des.append(drug_description)

    for idx, gene_pair in enumerate(fusion_df['Gene'].unique()):
        ncbi_text = "\n".join(gene_summaries)
        ckb_des = "\n".join(gene_ckb_des)
        drug_des= "\n".join(drug_des)
        add_gene_summary_text(document, gene_pair, ncbi_text, '', ckb_des, '', prognosis_des, '', '', drug_des, is_fusion=True)

In [23]:
def hrr_table(document, data):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(1.4)
        section.right_margin = Cm(2)

    table = document.add_table(rows=1, cols=2)
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(8.25)
    table.columns[1].width = Cm(8.25)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style = 'Table Grid'
    table.style.font.name = 'Times New Roman'
    table.style.font.size = Pt(11)

    header_row = table.rows[0].cells
    header_row[0].text = "Gene/Genomic Alteration"
    header_row[1].text = "Findings"
    for entry in data:
        row_cells = table.add_row().cells
        row_cells[0].text = entry.get("Gene/Genomic Alteration ", "")
        row_cells[1].text = entry.get("Finding ", "")
        
    for cell in table.rows[0].cells:
        cell.paragraphs[0].runs[0].font.name = 'Times New Roman'
        cell.paragraphs[0].runs[0].font.bold = True
        cell.paragraphs[0].runs[0].font.size = Pt(11)
        
    return table

In [24]:
def hrr_genes(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(12.17)
    new_height = Cm(0.85)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('HRR_genes.png', width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Homologous recombination repair (HRR) pathway genes play a vital role in maintaining genome stability and tumor suppression. Alterations in HRR genes can lead to genome instability which in turn may increase the risk of developing tumors. Thus knowing the alterations in HRR genes can act as potential biomarkers to decide the personalized therapy to be undertaken.')
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name ='Times New Roman'
    HRR_details = data.get('HRR_Details', '')
    hrr_data = json.loads(HRR_details)
    loh_percentages = [entry['Finding '] for entry in hrr_data if 'LOH percentage' in entry['Gene/Genomic Alteration ']]
    loh_percentage = loh_percentages[0].strip() if loh_percentages else '______'
    note_text = f'Note: This patient shows {loh_percentage} of LOH within the HRR gene panel that includes BRCA1/2. To further confirm the HRR/HRD status, assessment of HRR/HRD using orthogonal methods is recommended.'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    for char in note_text:
        run = p.add_run(char)
        run.font.size = Pt(11)
        run.font.name = 'Times New Roman'
        if char in loh_percentage:
            run.bold = True

    hrr_table(document, hrr_data)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('*Note: From the total of 46 HRR genes (including BRCA1 and BRCA2) covered in Oncomine Comprehensive Assay Plus, these alterations are Identified. Confirmation with other orthogonal methods is recommended.')
    for run in p.runs:
        run.font.size = Pt(10)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('*Note: Homologous recombination repair (HRR) genes were defined from published evidence in relevant therapies, clinical guidelines, as well as clinical trials, and include - BRCA1, BRCA2, ATM, BARD1, BLM, BRIP1, CDK12, CHEK1, CHEK2, FANCL, NBN, PALB2, POLD1, POLE, PPP2R2A, RAD51B, RAD51C, RAD51D, and RAD54L.')     
    for run in p.runs:
        run.font.size = Pt(10)
        run.font.name ='Times New Roman'

In [25]:
def clinical_trial(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(6.46)
    new_height = Cm(0.97)
    paragraph = document.add_paragraph()
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Clinical_trials.png', width=new_width, height=new_height)

In [26]:
def quality_control(document, data):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2)
        section.right_margin = Cm(2)

    table = document.add_table(rows=2, cols=2)
    table.style = 'Table Grid'
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(4.00)
    table.columns[1].width = Cm(4.00)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.style.font.name = 'Times New Roman'
    table.style.font.size = Pt(11)
    mean_depth_coverage = re.search(r'Average base coverage depth of (\d+)', data).group(1)
    target_base_coverage_500x = re.search(r'Target base coverage at 500X is (\d+\.\d+)%', data).group(1)
    table.cell(0, 0).text = 'Mean depth coverage'
    table.cell(0, 1).text = mean_depth_coverage
    table.cell(1, 0).text = 'Target Base coverage at 500X'
    table.cell(1, 1).text = f'{target_base_coverage_500x}%'
    return table

In [27]:
def supplementary_information(document, section):
    section.top_margin = Cm(2)
    section.bottom_margin = Cm(2)
    section.left_margin = Cm(0)
    section.right_margin = Cm(0)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Supplementary Information').bold=True
    for run in p.runs:
        run.font.size = Pt(14)
        run.font.name ='Times New Roman'
        run.font.underline = True
    p = document.add_paragraph()
    new_width = Cm(7.41)
    new_height = Cm(0.65)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Test_des.png', width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Comprehensive genomic profiling (CGP) is advancing precision oncology research through the analysis of multiple relevant biomarkers in a single next-generation sequencing (NGS) test. The test can detect all types of single-gene variants, such as single-nucleotide variants (SNVs), insertions and deletions (indels), novel and known fusions, splice variants, and copy number variants (CNVs), including both copy number gains and losses. A  study potential responses to immunotherapies with measurement of tumor mutational burden (TMB) and predisposition to genetic hypermutability by comparing microsatellite instability (MSI) regions, and analyse mutational signatures for insights into etiological factors in tumorigenesis. It can detect both gene-level and sample-level loss of heterozygosity (LOH) to assess genomic instability and mutations in 46 key genes in the homologous recombination repair (HRR) pathway.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 
        
    p = document.add_paragraph()
    new_width = Cm(7.41)
    new_height = Cm(0.71)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Quality_Control_Statics.png', width=new_width, height=new_height)
    sentence = data.get("QC statistics", " ")
    quality_control(document, sentence)
    
    p = document.add_paragraph()
    new_width = Cm(6.49)
    new_height = Cm(0.66)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Genes_analyzed.png', width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of DNA Sequence Variants: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABL1, ABL2, ACVR1, ACVR2A, AKT1, AKT2, AKT3, ALK, AMER1, APC, AR, ARAF, ARID1A, ARID1B, ARID2, ASXL1, ASXL2, ATM, ATP1A1, ATR, ATRX, AURKA, AURKC, AXIN1, AXIN2, AXL, B2M, BAP1, BCL2, BCL2L12, BCL6, BCOR, BCR, BLM, BMP5, BRAF, BRCA1, BRCA2, BRIP1, BTK, CACNA1D, CALR, CARD11, CASP8, CBL, CCND1, CCND2, CCND3, CCNE1, CD79B, CDC73, CDH1, CDK4, CDK6, CDKN2A, CDKN2C, CHD4, CHEK2, CIC, CREBBP, CSF1R, CTCF, CTNNB1, CUL1, CUL3, CYP2D6, CYSLTR2, DDR2, DDX3X, DGCR8, DICER1, DNMT3A, DPYD, DROSHA, E2F1, EGFR, EIF1AX, EP300, EPAS1, EPHA2, ERBB2, ERBB3, ERBB4, ERCC2, ERCC5, ERRFI1, ESR1, EZH2, FAM135B, FANCM, FBXW7, FGF7, FGFR1, FGFR2, FGFR3, FGFR4, FLT3, FLT4, FOXA1, FOXL2, FOXO1, FUBP1, GATA2, GATA3, GLI1, GNA11, GNA13, GNAQ, GNAS, GPS2, H2BC5, H3-3A, H3-3B, H3C2, HIF1A, HNF1A, HRAS, ID3, IDH1, IDH2, IKBKB, IL6ST, IL7R, IRF4, IRS4, JAK1, JAK2, JAK3, KDM6A, KDR, KEAP1, KIT, KLF4, KLF5, KMT2B, KMT2D, KNSTRN, KRAS, LARP4B, LATS1, MAGOH, MAP2K1, MAP2K2, MAP2K4, MAP2K7, MAP3K4, MAPK1, MAPK8, MAX, MDM4, MECOM, MED12, MEF2B, MEN1, MET, MGA, MITF, MLH3, MPL, MSH3, MSH6, MTOR, MYC, MYCN, MYD88, MYOD1, NBN, NCOR1, NF1, NF2, NFE2L2, NOTCH1, NOTCH2, NRAS, NSD2, NT5C2, NTRK1, NTRK2, NTRK3, NUP93, PARP1, PAX5, PBRM1, PCBP1, PDGFRA, PDGFRB, PHF6, PIK3C2B, PIK3CA, PIK3CB, PIK3CD, PIK3CG, PIK3R1, PIK3R2, PIM1, PLCG1, PMS2, POLE, PPM1D, PPP2R1A, PPP6C, PRKACA, PTCH1, PTEN, PTPN11, PTPRD, PXDNL, RAC1, RAD50, RAD51, RAF1, RARA, RB1, RET, RGS7, RHEB, RHOA, RICTOR, RIT1, RNF43, ROS1, RPL10, RPL5, RUNX1, RUNX1T1, SDHD, SETBP1, SETD2, SF3B1, SIX1, SIX2, SLCO1B3, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SMC1A, SMO, SNCAIP, SOCS1, SOS1, SOX2, SPOP, SRC, SRSF2, STAG2, STAT3, STAT5B, STAT6, STK11, TAF1, TCF7L2, TERT, TET2, TGFBR1, TGFBR2, TNFAIP3, TOP1, TP53, TPMT, TRRAP, TSC2, TSHR, U2AF1, UGT1A1, USP8, VHL, WAS, WT1, XPO1, XRCC2, ZFHX3, ZNF217, ZNF429')
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)   
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of Copy Number Variations: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABCB1, ABL1, ABL2, ABRAXAS1, ACVR1B, ACVR2A, ADAMTS12, ADAMTS2, AKT1, AKT2, AKT3, ALK, AMER1, APC, AR, ARAF, ARHGAP35, ARID1A, ARID1B, ARID2, ARID5B, ASXL1, ASXL2, ATM, ATR, ATRX, AURKA, AURKC, AXIN1, AXIN2, AXL, B2M, BAP1, BARD1, BCL2, BCL2L12, BCL6, BCOR, BLM, BMPR2, BRAF, BRCA1, BRCA2, BRIP1, CARD11, CASP8, CBFB, CBL, CCND1, CCND2, CCND3, CCNE1, CD274, CD276, CDC73, CDH1, CDH10, CDK12, CDK4, CDK6, CDKN1A, CDKN1B, CDKN2A, CDKN2B, CDKN2C, CHD4, CHEK1, CHEK2, CIC, CREBBP, CSMD3, CTCF, CTLA4, CTNND2, CUL3, CUL4A, CUL4B, CYLD, CYP2C9, DAXX, DDR1, DDR2, DDX3X, DICER1, DNMT3A, DOCK3, DPYD, DSC1, DSC3, EGFR, EIF1AX, ELF3, EMSY, ENO1, EP300, EPCAM, EPHA2, ERAP1, ERAP2, ERBB2, ERBB3, ERBB4, ERCC2, ERCC4, ERRFI1, ESR1, ETV6, EZH2, FAM135B, FANCA, FANCC, FANCD2, FANCE, FANCF, FANCG, FANCI,  FANCL, FANCM, FAT1, FBXW7, FGF19, FGF23, FGF3, FGF4, FGF9, FGFR1, FGFR2, FGFR3, FGFR4, FLT3, FLT4, FOXA1, FUBP1, FYN, GATA2, GATA3, GLI3, GNA13, GNAS, GPS2, H3-3A, H3-3B, HDAC2, HDAC9, HLA-A, HLA-B, HNF1A, IDH2, IGF1R, IKBKB, IL7R, INPP4B, JAK1, JAK2, JAK3, KDM5C, KDM6A, KDR, KEAP1, KIT, KLF5, KMT2A, KMT2B, KMT2C, KMT2D, KRAS, LARP4B, LATS1, LATS2, MAGOH, MAP2K1, MAP2K4, MAP2K7, MAP3K1, MAP3K4, MAPK1, MAPK8, MAX, MCL1, MDM2, MDM4, MECOM, MEF2B, MEN1, MET, MGA, MITF, MLH1, MLH3, MPL, MRE11, MSH2, MSH3, MSH6, MTAP, MTOR, MUTYH, MYC, MYCL, MYCN, MYD88, NBN, NCOR1, NF1, NF2, NFE2L2, NOTCH1, NOTCH2, NOTCH3, NOTCH4, NRAS, NTRK1, NTRK3, PALB2, PARP1, PARP2, PARP3, PARP4, PBRM1, PCBP1, PDCD1, PDCD1LG2, PDGFRA, PDGFRB, PDIA3, PGD, PHF6, PIK3C2B, PIK3CA, PIK3CB, PIK3R1, PIK3R2, PIM1, PLCG1, PMS1, PMS2, POLD1, POLE, POT1, PPM1D, PPP2R1A, PPP2R2A, PPP6C, PRDM1, PRDM9, PRKACA, PRKAR1A, PTCH1, PTEN, PTPN11, PTPRT, PXDNL, RAC1, RAD50, RAD51, RAD51B, RAD51C, RAD51D, RAD52, RAD54L, RAF1, RARA, RASA1, RASA2, RB1, RBM10, RECQL4, RET, RHEB, RICTOR, RIT1, RNASEH2A, RNASEH2B, RNF43, ROS1, RPA1, RPS6KB1, RPTOR, RUNX1, SDHA, SDHB, SDHD, SETBP1, SETD2, SF3B1, SLCO1B3, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SMC1A, SMO, SOX9, SPEN, SPOP, SRC, STAG2, STAT3, STAT6, STK11, SUFU, TAP1, TAP2, TBX3, TCF7L2, TERT, TET2, TGFBR2, TNFAIP3, TNFRSF14, TOP1, TP53, TP63, TPMT, TPP2, TSC1, TSC2, U2AF1, USP8, USP9X, VHL, WT1, XPO1, XRCC2, XRCC3, YAP1, YES1, ZFHX3, ZMYM3, ZNF217, ZNF429, ZRSR2')
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed for the Detection of Fusions: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('AKT1, AKT2, AKT3, ALK, AR, BRAF, BRCA1, CDKN2A, EGFR, ERBB2, ERBB4, ERG, ESR1, ETV1, ETV4, ETV5, FGFR1, FGFR2, FGFR3, MAP3K8, MET, MTAP, MYB, MYBL1, NOTCH1, NOTCH2, NOTCH3, NRG1, NTRK1, NTRK2, NTRK3, NUTM1, PIK3CA, PIK3CB, PPARG, PRKACA, PRKACB, RAF1, RARA, RELA, RET, ROS1, RSPO2, RSPO3, STAT6, TERT, TFE3, TFEB, YAP1') 
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    header_run = p.add_run('Genes Assayed with Full Exon Coverage: ')
    header_run.bold = True
    header_run.underline = True
    header_run.font.name = 'Times New Roman'
    header_run.font.size = Pt(9)
    genes_text = ('ABRAXAS1, ACVR1B, ACVR2A, ADAMTS12, ADAMTS2, AMER1, APC, ARHGAP35, ARID1A, ARID1B, ARID2, ARID5B, ASXL1, ASXL2, ATM, ATR, ATRX, AXIN1, AXIN2, B2M, BAP1, BARD1, BCOR, BLM, BMPR2, BRCA1, BRCA2, BRIP1, CALR, CASP8, CBFB, CD274, CD276, CDC73, CDH1, CDH10, CDK12, CDKN1A, CDKN1B, CDKN2A, CDKN2B, CDKN2C, CHEK1, CHEK2, CIC, CIITA, CREBBP, CSMD3, CTCF, CTLA4, CUL3, CUL4A, CUL4B, CYLD, CYP2C9, CYP2D6, DAXX, DDX3X, DICER1, DNMT3A, DOCK3, DPYD, DSC1, DSC3, ELF3, ENO1, EP300, EPCAM, EPHA2, ERAP1, ERAP2, ERCC2, ERCC4, ERCC5, ERRFI1, ETV6, FANCA, FANCC, FANCD2, FANCE, FANCF, FANCG, FANCI, FANCL, FANCM, FAS, FAT1, FBXW7, FUBP1, GATA3, GNA13, GPS2, HDAC2, HDAC9, HLA-A, HLA-B, HNF1A, ID3, INPP4B, JAK1, JAK2, JAK3, KDM5C, KDM6A, KEAP1, KLHL13, KMT2A, KMT2B, KMT2C, KMT2D, LARP4B, LATS1, LATS2, MAP2K4, MAP2K7, MAP3K1, MAP3K4, MAPK8, MEN1, MGA, MLH1, MLH3, MRE11, MSH2, MSH3, MSH6, MTAP, MTUS2, MUTYH, NBN, NCOR1, NF1, NF2, NOTCH1, NOTCH2, NOTCH3, NOTCH4, PALB2, PARP1, PARP2, PARP3, PARP4, PBRM1, PDCD1, PDCD1LG2, PDIA3, PGD, PHF6, PIK3R1, PMS1, PMS2, POLD1, POLE, POT1, PPM1D, PPP2R2A, PRDM1, PRDM9, PRKAR1A, PSMB10, PSMB8, PSMB9, PTCH1, PTEN, PTPRT, RAD50, RAD51, RAD51B, RAD51C, RAD51D, RAD52, RAD54L, RASA1, RASA2, RB1, RBM10, RECQL4, RNASEH2A, RNASEH2B, RNASEH2C, RNF43, RPA1, RPL22, RPL5, RUNX1, RUNX1T1, SDHA, SDHB, SDHC, SDHD, SETD2, SLX4, SMAD2, SMAD4, SMARCA4, SMARCB1, SOCS1, SOX9, SPEN, STAG2, STAT1, STK11, SUFU, TAP1, TAP2, TBX3, TCF7L2, TET2, TGFBR2, TMEM132D, TNFAIP3, TNFRSF14, TP53, TP63, TPP2, TSC1, TSC2, UGT1A1, USP9X, VHL, WT1, XRCC2, XRCC3, ZBTB20, ZFHX3, ZMYM3, ZRSR2') 
    genes_run = p.add_run(genes_text)
    genes_run.italic = True
    genes_run.font.name = 'Times New Roman'
    genes_run.font.size = Pt(9)
    
    p = document.add_paragraph()
    new_width = Cm(6.85)
    new_height = Cm(0.69)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Methodology.png', width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('Nucleic acid (DNA/RNA) was extracted from FFPE tissue samples, using standard Qiagen nucleic acid isolation kits. The DNA/RNA was quantified using Qubit and 20-30 ng of DNA/RNA was amplified using Oncomine Comprehensive assay plus as per the instruction manual. The QC of the library prepared is checked by Agilent Bioanalyser. The 150-200 bp library size samples are sequenced on Ion S5 platform as per manufacturer protocol.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 
        
    p = document.add_paragraph()
    new_width = Cm(7.14)
    new_height = Cm(0.69)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Data_analysis.png', width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The sequence data is processed using the analysis pipeline Ion reporter software 5.18.2.0 as suggested by the vendor. This software can detects and annotates low frequency somatic variants (SNPs, InDels, CNVs) from targeted Ion AmpliSeq™ DNA manual or Ion Chef™ automated libraries, computes automatic tumor cellularity, Loss-of-Heterozygosity, TMB and microsatellite instability (MSI), as well as gene fusions from targeted Ion AmpliSeq™ RNA manual or Ion Chef™ automated libraries, from Oncomine™ Comprehensive Assay Plus run on the Ion 540™ Chip. TMB uses the TMB (Non-Germline Mutations) filter chain with TMB algorithm v4.0. MSI status is computed using a baseline with MSI algorithm v4.0.3. Released with: Ion Reporter™ Software 5.18.4. Workflow Version: 2.4. Samples are classified as TMB-High or TMB-low using a cutoff value of 10mut/mb.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 
        
    p = document.add_paragraph()
    new_width = Cm(8.23)
    new_height = Cm(0.61)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Interpretation_of_variants.png', width=new_width, height=new_height)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The interpretation of the report is based on the clinical information provided. The annotation for variants was derived using various disease databases like dbSNP, ClinVar. The population frequency information from 1000 genomes, ExAC, GnomAD was used for the elimination of common variants/polymorphism. For prediction of the possible impact of coding non-synonymous SNVs on the structure and function of protein, PolyPhen-2, MT2 and SIFT score was used. Further Oncomine Reporter software was used for annotating variants with a curated list of relevant labels, guidelines, and global clinical trials. Oncomine Comprehensive genomic profiling assay will analyze across >500 genes(SNVs,Indels,CNVs,Fusions), plus key immuno-oncology research biomarkers like tumor mutational burden (TMB), microsatellite instability (MSI), and Homologous Recombination Repair genes (HRR).') 
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman'
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The variant is reported as per the Standards and Guidelines for the Interpretation and Reporting of Sequence Variants in Cancer- A Joint Consensus Recommendation of the Association for Molecular Pathology, American Society of Clinical Oncology, and College of American Pathologists.')
    for run in p.runs:
        run.font.size = Pt(11)
        run.font.name ='Times New Roman' 

In [28]:
def supplementary_table(document, section):
    section.top_margin = Cm(2)
    section.bottom_margin = Cm(2)
    section.left_margin = Cm(2)
    section.right_margin = Cm(2)
    p = document.add_paragraph()
    table = doc.add_table(rows=11, cols=2)
    table.style = 'Table Grid'
    table.autofit = False
    table.allow_autofit = False
    table.columns[0].width = Cm(3.31)
    table.columns[1].width = Cm(13.36)
    table.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    table.cell(0, 0).merge(table.cell(0, 1)).text = 'Variant Classification'
    table.cell(0, 0).paragraphs[0].runs[0].bold = True
    table.cell(1, 0).merge(table.cell(1, 1)).text = 'Tier I: Variants of Strong clinical significance'
    table.cell(1, 0).paragraphs[0].runs[0].bold = True
    table.cell(2, 0).text = 'Level A'
    table.cell(2, 0).paragraphs[0].runs[0].bold = True
    table.cell(2, 1).text = 'FDA-approved therapy included in professional guidelines'
    table.cell(3, 0).text = 'Level B'
    table.cell(3, 0).paragraphs[0].runs[0].bold = True
    table.cell(3, 1).text = 'Well-powered studies with consensus from experts in the field'
    table.cell(4, 0).merge(table.cell(4, 1)).text = 'Tier II: Variants of Potential Clinical Significance'
    table.cell(4, 0).paragraphs[0].runs[0].bold = True
    table.cell(5, 0).text = 'Level C'
    table.cell(5, 0).paragraphs[0].runs[0].bold = True
    table.cell(5, 1).text = 'FDA-approved therapies for different tumor types or investigational therapies\nMultiple small published studies with some consensus'
    table.cell(6, 0).text = 'Level D'
    table.cell(6, 0).paragraphs[0].runs[0].bold = True
    table.cell(6, 1).text = 'Preclinical trials or a few case reports without consensus'
    table.cell(7, 0).merge(table.cell(7, 1)).text = 'Tier III: Variants of Unknown Clinical Significance'
    table.cell(7, 0).paragraphs[0].runs[0].bold = True
    table.cell(8, 0).merge(table.cell(8, 1)).text = 'Not observed at a significant allele frequency in the general or specific subpopulation databases, or pan-cancer or tumor-specific variant databases. No convincing published evidence of cancer association'
    table.cell(9, 0).merge(table.cell(9, 1)).text = 'Tier IV: Benign or Likely Benign Variants'
    table.cell(9, 0).paragraphs[0].runs[0].bold = True
    table.cell(10, 0).merge(table.cell(10, 1)).text = 'Observed at a significant allele frequency in the general or specific subpopulation databases. No existing published evidence of cancer association'
    for row in table.rows:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                if paragraph.runs:
                    run = paragraph.runs[0]
                    run.font.name = 'Times New Roman'
                    run.font.size = Pt(10)
    p = document.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    p.paragraph_format.left_indent = Cm(2.5)
    p.paragraph_format.right_indent = Cm(2.5)
    p.add_run('The transcript used for clinical reporting generally represents the canonical transcript (according to the Ensembl release 37 gene model), which is usually the longest coding transcript with strong/multiple supporting evidence. However, clinically relevant variants annotated in alternate complete coding transcripts could also be reported. The in silico predictions are based on Variant Effect Predictor, Ensembl release 91 (SIFT version - 5.2.2; PolyPhen - 2.2.2); 2019 release from dbNSFPv4.0 and MutationTaster2 based on build NCBI/ Ensembl 66.')
    for run in p.runs:
        run.font.size = Pt(10)
        run.font.name ='Times New Roman' 

In [29]:
def disclaimer(document):
    for section in document.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(0)
        section.right_margin = Cm(0)
    new_width = Cm(6.00)
    new_height = Cm(0.7)
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.left_indent = 0
    run = paragraph.add_run()
    run.add_picture('Disclaimer.png', width=new_width, height=new_height)
    
    bullet_points = ['This report is based on the sample received in the Lilac Insights laboratory; the analysis is based on the assumption that samples received are representative of the patient demographics mentioned on the test requisition form and the sample tube.',
        'Despite all the necessary precautions and stringency adopted whilst performing DNA tests, the currently available data indicates that the technical error rate associated with all types of DNA analysis, is approximately 2%.',
        'As with all diagnostic tests, the laboratory report must be interpreted in conjunction with the presenting clinical profile of the individual tested and evaluation of all reports. Interpretation of variants is performed based on the current knowledge standards and reporting guidelines. In cases of presence of a VUS, we recommend periodic review of these variants to determine any change in classification based on new published research.',
        'The classification and interpretation of all the variants is carried out based on the current state of scientific knowledge and medical understanding. The results should be correlated clinically.',
        'This report cannot be used for medico legal purposes.']

    for point in bullet_points:
        add_bullet_point(document.add_paragraph(), point)

def add_bullet_point(paragraph, text):
    run = paragraph.add_run(u'\u2022 ') 
    font = run.font
    font.name = 'Times New Roman'
    font.size = Pt(10)
    paragraph.add_run(text)
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY
    paragraph.paragraph_format.left_indent = Cm(2.5)
    paragraph.paragraph_format.right_indent = Cm(2.5)

In [30]:
doc = Document()
add_header_footer(doc)
add_internal_reference_table(doc)
patient_information(doc)
oncoprecise_image(doc)
clinical_history(doc)
report_summary(doc)
therapeutic_summary(doc)
immunotherapy_markers(doc)
approved_immunotherapy(doc)
intrial_immunotherapy(doc)
variant_details(doc)
genes_involved = []
gene_varient_description(doc, genes_involved)
hrr_genes(doc)
clinical_trial(doc)
section1 = doc.sections[0]
supplementary_information(doc, section1)
if len(doc.sections) < 2:
    doc.add_section()
    section2 = doc.sections[1]
    supplementary_table(doc, section2)
disclaimer(doc)
doc.save('report.docx')

clear_output(wait=True)
message = "<p style='font-size: 24px; font-weight: bold;'>REPORT IS READY FOR DOWNLOAD</p>"
display(Markdown(message))
download_link = f"<a href='report.docx' download='report.docx'>Download the Report</a>"
display(HTML(download_link))

<p style='font-size: 24px; font-weight: bold;'>REPORT IS READY FOR DOWNLOAD</p>